<a href="https://colab.research.google.com/github/Chunsen41/Chunsen41/blob/main/TestCode_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROBUST LINKEDIN + INDEED COLLECTOR
# Tries site: filters + fallbacks (with/without location)
# Filters by domain when site: is omitted
# Uses Indeed RSS as backup
# Saves to SQLite and shows preview

In [ ]:
# CONFIGURATIONS
SERPAPI_API_KEY = "" # YOU PUT YOUR API KEY HERE!!!!!!!

# Role/location pairs to try
QUERIES = [
    ("Data Analyst", "New York, NY", 30),
    ("Data Engineer remote", "United States", 30),
]

# Whether to include generic (no site:) attempts then filter by domain
TRY_GENERIC_AFTER_SITE = True

# Locale hints (can help coverage)
SERPAPI_HL = "en"   # UI language
SERPAPI_GL = "us"   # country

# Indeed RSS backups (US region)
INDEED_RSS = [
    "https://www.indeed.com/rss?q=Data+Analyst&l=New+York%2C+NY",
    "https://www.indeed.com/rss?q=Data+Engineer&l=United+States",
]

DB_PATH  = "jobs_li_indeed_robust.db"
MAX_SHOW = 50

#  DEPS
import sys, subprocess, importlib
def _pip(pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs, "-q"])
need=[]
for m,p in [("pandas","pandas"),("requests","requests"),("feedparser","feedparser")]:
    try: importlib.import_module(m)
    except ImportError: need.append(p)
if need: _pip(need)

# IMPORTS
import re, os, sqlite3, requests, feedparser, pandas as pd
from datetime import datetime, timezone

CANON = ["post_date","company","job_title","position","location","job_url","source","description"]

def _clean(s): return re.sub(r"\s+"," ", str(s)).strip() if s is not None else None
def _row(**kw):
    r = {k: _clean(kw.get(k)) for k in CANON}
    if r["position"] is None: r["position"] = r["job_title"]
    return r

# SerpAPI Google Jobs
def _serpapi_call(q, location=None, num=25, api_key=SERPAPI_API_KEY, hl=SERPAPI_HL, gl=SERPAPI_GL):
    params = {"engine":"google_jobs", "q": q, "num": num, "api_key": api_key, "hl": hl, "gl": gl}
    if location and location.lower() != "remote":
        params["location"] = location
    r = requests.get("https://serpapi.com/search.json", params=params, timeout=40)
    return r, params

def _parse_jobs_json(j):
    out=[]
    for job in j.get("jobs_results", []) or []:
        det  = job.get("detected_extensions", {}) or {}
        link = job.get("apply_options", [{}])[0].get("link") if job.get("apply_options") else job.get("link")
        out.append(_row(
            post_date   = job.get("published_date") or det.get("posted_at"),
            company     = job.get("company_name"),
            job_title   = job.get("title"),
            position    = job.get("title"),
            location    = job.get("location"),
            job_url     = link,
            source      = "GoogleJobs (SerpAPI)",
            description = (job.get("description") or "")[:2000]
        ))
    return out

def fetch_domain(role, location, num, domain):
    """
    Try multiple permutations to get jobs from a target domain (linkedin.com or indeed.com).
    Strategy:
      1 q = 'site:<domain> <role>' with location
      2 same q with location='United States'
      3 same q with no location
      4 q = '<role>' generic (with/without location), then filter rows by domain
    """
    rows = []
    attempts = []

    def attempt(q, loc):
        r, p = _serpapi_call(q, loc, num)
        attempts.append((r.status_code, p))
        if r.status_code == 200:
            try: data = r.json()
            except Exception: return []
            return _parse_jobs_json(data)
        # on 400 unsupported location, retry without location
        if r.status_code == 400 and "Unsupported" in r.text:
            r2, p2 = _serpapi_call(q, None, num)
            attempts.append((r2.status_code, p2))
            if r2.status_code == 200:
                try: data = r2.json()
                except Exception: return []
                return _parse_jobs_json(data)
        return []

    # 1 site:domain with original location
    q1 = f"site:{domain} {role}"
    rows += attempt(q1, location)

    # 2 site:domain with US location, if different
    if (location or "").lower() != "united states":
        rows += attempt(q1, "United States")

    # 3 site:domain with no location
    rows += attempt(q1, None)

    # 4 generic (optional), then filter by domain
    if TRY_GENERIC_AFTER_SITE and not rows:
        qg = role
        rows += attempt(qg, location)
        rows += attempt(qg, "United States")
        rows += attempt(qg, None)
        # filter by domain
        rows = [r for r in rows if r.get("job_url") and domain in r["job_url"]]

    # de-dup
    uniq = {}
    for r in rows:
        u = r.get("job_url")
        if u and u not in uniq: uniq[u] = r
    out = list(uniq.values())

    # log attempts
    print(f"[{domain}] Attempts: " + " | ".join([f"{s} {p.get('q')} [{p.get('location','no-loc')}]" for s,p in attempts]))
    print(f"[{domain}] Found: {len(out)}")
    return pd.DataFrame(out, columns=CANON) if out else pd.DataFrame(columns=CANON)

#  Indeed
def fetch_indeed_rss(url):
    try:
        feed = feedparser.parse(url)
    except Exception as e:
        print("[Indeed RSS] Error:", e)
        return pd.DataFrame(columns=CANON)
    rows=[]
    for e in feed.entries:
        rows.append(_row(
            post_date   = e.get("published") or e.get("updated"),
            company     = None,
            job_title   = e.get("title"),
            position    = e.get("title"),
            location    = None,
            job_url     = e.get("link"),
            source      = "Indeed RSS",
            description = e.get("summary") or ""
        ))
    return pd.DataFrame(rows, columns=CANON)

# SQLite THE DATABASE
def init_db(path=DB_PATH):
    conn = sqlite3.connect(path)
    conn.execute("""CREATE TABLE IF NOT EXISTS jobs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        job_url TEXT UNIQUE,
        post_date TEXT, company TEXT, job_title TEXT, position TEXT, location TEXT,
        source TEXT, description TEXT, added_at TEXT
    )""")
    conn.commit(); return conn

def upsert(conn, df):
    cur = conn.cursor(); ins=upd=0
    for _, r in df.iterrows():
        url = r.get("job_url")
        if not url: continue
        now = datetime.now(timezone.utc).isoformat()
        try:
            cur.execute("""INSERT INTO jobs(job_url,post_date,company,job_title,position,location,source,description,added_at)
                           VALUES (?,?,?,?,?,?,?,?,?)""",
                        (url, r.post_date, r.company, r.job_title, r.position, r.location, r.source, r.description, now))
            ins += 1
        except sqlite3.IntegrityError:
            upd += 1
    conn.commit(); return ins, upd

#  RUN IT HERE
frames = []

for role, loc, n in QUERIES:
    print(f"\n=== ROLE='{role}' | LOCATION='{loc}' | NUM={n} ===")
    # LinkedIn
    df_li = fetch_domain(role, loc, n, domain="linkedin.com")
    if not df_li.empty:
        df_li["source"] = "LinkedIn (via GoogleJobs/SerpAPI)"
        frames.append(df_li)

    # Indeed
    df_in = fetch_domain(role, loc, n, domain="indeed.com")
    if not df_in.empty:
        df_in["source"] = "Indeed (via GoogleJobs/SerpAPI)"
        frames.append(df_in)

# RSS backups
for rss in INDEED_RSS:
    print(f"\n[RSS] {rss}")
    df_rss = fetch_indeed_rss(rss)
    if not df_rss.empty:
        frames.append(df_rss)

if not frames:
    print("\nNo results from LinkedIn/Indeed paths. Try a broader role, remove location, or increase num.")
else:
    df_all = pd.concat(frames, ignore_index=True)[CANON]
    df_all = df_all.drop_duplicates(subset=["job_url"], keep="first")
    print(f"\nCombined unique rows: {len(df_all)}")
    display(df_all.head(MAX_SHOW))

    conn = init_db(DB_PATH)
    ins, upd = upsert(conn, df_all)
    print(f"SQLite → inserted={ins}, updated={upd}")

    preview = pd.read_sql_query(
        "SELECT post_date,company,job_title,position,location,job_url,source,added_at "
        "FROM jobs ORDER BY added_at DESC LIMIT ?", conn, params=(MAX_SHOW,))
    display(preview)
    conn.close()



=== ROLE='Data Analyst' | LOCATION='New York, NY' | NUM=30 ===
[linkedin.com] Attempts: 200 site:linkedin.com Data Analyst [New York, NY] | 200 site:linkedin.com Data Analyst [United States] | 200 site:linkedin.com Data Analyst [no-loc] | 200 Data Analyst [New York, NY] | 200 Data Analyst [United States] | 200 Data Analyst [no-loc]
[linkedin.com] Found: 2
[indeed.com] Attempts: 200 site:indeed.com Data Analyst [New York, NY] | 200 site:indeed.com Data Analyst [United States] | 200 site:indeed.com Data Analyst [no-loc] | 200 Data Analyst [New York, NY] | 200 Data Analyst [United States] | 200 Data Analyst [no-loc]
[indeed.com] Found: 6

=== ROLE='Data Engineer remote' | LOCATION='United States' | NUM=30 ===
[linkedin.com] Attempts: 200 site:linkedin.com Data Engineer remote [United States] | 200 site:linkedin.com Data Engineer remote [no-loc] | 200 Data Engineer remote [United States] | 200 Data Engineer remote [United States] | 200 Data Engineer remote [no-loc]
[linkedin.com] Found: 0

,post_date,company,job_title,position,location,job_url,source,description
0,3 days ago,Capital Rx,Network Pricing Data Analyst,Network Pricing Data Analyst,Anywhere,https://www.linkedin.com/jobs/view/network-pri...,LinkedIn (via GoogleJobs/SerpAPI),About Capital Rx Capital Rx is a health techno...
1,9 hours ago,Harvey Nash,"Data Analyst - New Haven, CT (Hybrid)","Data Analyst - New Haven, CT (Hybrid)","New Haven, CT",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn (via GoogleJobs/SerpAPI),"Job Title: Data Analyst Location: New Haven, C..."
2,6 days ago,Montefiore Einstein,Data Analyst,Data Analyst,"Bronx, NY",https://www.indeed.com/viewjob?jk=c4cf7b0726f8...,Indeed (via GoogleJobs/SerpAPI),"City/State: Bronx, New York Grant Funded: Yes ..."
3,5 days ago,Guardian Life Insurance Company,Customer Data Management Analyst,Customer Data Management Analyst,"New York, NY",https://www.indeed.com/viewjob?jk=772eb7ff6af0...,Indeed (via GoogleJobs/SerpAPI),"Every day, Guardian helps our 29 million custo..."
4,None,PayPal,"Analyst, Data Analytics","Analyst, Data Analytics","New York, NY",https://www.indeed.com/viewjob?jk=65d367fd99ff...,Indeed (via GoogleJobs/SerpAPI),The Company PayPal has been revolutionizing co...
5,None,Meta,Data Analyst - People Analytics,Data Analyst - People Analytics,"New York, NY",https://www.indeed.com/viewjob?jk=7a36f733e4fc...,Indeed (via GoogleJobs/SerpAPI),Data Analysts in People Analytics drive the bu...
6,4 days ago,PAYLOCITY CORPORATION,Senior Sales Data Analyst,Senior Sales Data Analyst,Anywhere,https://www.indeed.com/viewjob?jk=e24f0df2290a...,Indeed (via GoogleJobs/SerpAPI),Company Overview Paylocity is an award-winning...
7,6 days ago,Cellhub,Data Analyst - Junior,Data Analyst - Junior,"Hicksville, NY",https://www.indeed.com/viewjob?jk=2b85e3399082...,Indeed (via GoogleJobs/SerpAPI),Job Overview We are seeking a detail-oriented ...
8,None,Scalepex,AWS Data Engineer - Fully Remote - US Only,AWS Data Engineer - Fully Remote - US Only,Anywhere,https://www.indeed.com/viewjob?jk=ef73e4bc9fad...,Indeed (via GoogleJobs/SerpAPI),❋ Why Scalepex? Scalepex is a dynamic services...


SQLite → inserted=0, updated=9


,post_date,company,job_title,position,location,job_url,source,added_at
0,None,Scalepex,AWS Data Engineer - Fully Remote - US Only,AWS Data Engineer - Fully Remote - US Only,Anywhere,https://www.indeed.com/viewjob?jk=ef73e4bc9fad...,Indeed (via GoogleJobs/SerpAPI),2025-09-22T22:37:22.124258+00:00
1,6 days ago,Cellhub,Data Analyst - Junior,Data Analyst - Junior,"Hicksville, NY",https://www.indeed.com/viewjob?jk=2b85e3399082...,Indeed (via GoogleJobs/SerpAPI),2025-09-22T22:37:22.124153+00:00
2,4 days ago,PAYLOCITY CORPORATION,Senior Sales Data Analyst,Senior Sales Data Analyst,Anywhere,https://www.indeed.com/viewjob?jk=e24f0df2290a...,Indeed (via GoogleJobs/SerpAPI),2025-09-22T22:37:22.124035+00:00
3,None,Meta,Data Analyst - People Analytics,Data Analyst - People Analytics,"New York, NY",https://www.indeed.com/viewjob?jk=7a36f733e4fc...,Indeed (via GoogleJobs/SerpAPI),2025-09-22T22:37:22.123925+00:00
4,None,PayPal,"Analyst, Data Analytics","Analyst, Data Analytics","New York, NY",https://www.indeed.com/viewjob?jk=65d367fd99ff...,Indeed (via GoogleJobs/SerpAPI),2025-09-22T22:37:22.123808+00:00
5,5 days ago,Guardian Life Insurance Company,Customer Data Management Analyst,Customer Data Management Analyst,"New York, NY",https://www.indeed.com/viewjob?jk=772eb7ff6af0...,Indeed (via GoogleJobs/SerpAPI),2025-09-22T22:37:22.123689+00:00
6,6 days ago,Montefiore Einstein,Data Analyst,Data Analyst,"Bronx, NY",https://www.indeed.com/viewjob?jk=c4cf7b0726f8...,Indeed (via GoogleJobs/SerpAPI),2025-09-22T22:37:22.123537+00:00
7,9 hours ago,Harvey Nash,"Data Analyst - New Haven, CT (Hybrid)","Data Analyst - New Haven, CT (Hybrid)","New Haven, CT",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn (via GoogleJobs/SerpAPI),2025-09-22T22:37:22.123280+00:00
8,3 days ago,Capital Rx,Network Pricing Data Analyst,Network Pricing Data Analyst,Anywhere,https://www.linkedin.com/jobs/view/network-pri...,LinkedIn (via GoogleJobs/SerpAPI),2025-09-22T22:37:22.122772+00:00


# Fetch 75 jobs (25 per source)
# Sources:
#   - LinkedIn (via GoogleJobs/SerpAPI with site filter)
#   - Indeed  (via GoogleJobs/SerpAPI with site filter)
#   - Google Jobs (generic; excludes linkedin/indeed URLs)
# Save to CSV and auto-download in Colab

In [ ]:
#  SETTINGS
TARGET_PER_SOURCE = 25                  # 25 each => total 75
CSV_PATH = None                         # auto-name if None (based on role + timestamp)
ROLE_FOR_CSV_NAME = "Data Analyst"      # just for filename prettiness

# ---- uses your existing config/funcs from the notebook ----
# expects:
#   SERPAPI_API_KEY, QUERIES, TRY_GENERIC_AFTER_SITE, SERPAPI_HL, SERPAPI_GL
#   _serpapi_call(q, location, num, api_key, hl, gl)
#   _parse_jobs_json(json) -> list of canonical dicts
#   fetch_domain(role, location, num, domain) -> DataFrame(CANON)
#   CANON with canonical columns and _row/_clean helpers
#   init_db, upsert, DB_PATH, MAX_SHOW

import pandas as pd, re
from datetime import datetime

# -- generic Google Jobs fetcher (no site:), filtered to exclude linkedin/indeed so it's a third bucket
def fetch_generic(role, location, num=25):
    rows = []

    # try with requested location
    r, p = _serpapi_call(role, location, num)
    if r.status_code == 200:
        try:
            rows += _parse_jobs_json(r.json())
        except Exception:
            pass
    else:
        # retry without location (handles unsupported locations)
        r2, p2 = _serpapi_call(role, None, num)
        if r2.status_code == 200:
            try:
                rows += _parse_jobs_json(r2.json())
            except Exception:
                pass

    if not rows:
        return pd.DataFrame(columns=CANON)

    df = pd.DataFrame(rows, columns=CANON)
    # Exclude linkedin/indeed so this bucket stays "generic"
    mask = ~(df["job_url"].str.contains("linkedin.com", na=False) | df["job_url"].str.contains("indeed.com", na=False))
    df = df[mask].copy()
    df["source"] = "GoogleJobs (generic via SerpAPI)"
    # de-dup within this set
    if "job_url" in df.columns:
        df = df.drop_duplicates(subset=["job_url"], keep="first")
    return df.head(num)

def _accumulate_until(df_current, df_new, needed):
    """
    Add rows from df_new into df_current until 'needed' count is reached (dedup by job_url).
    Returns updated df_current and the remaining 'needed'.
    """
    if df_new.empty or needed <= 0:
        return df_current, needed
    # ensure columns
    for c in CANON:
        if c not in df_new.columns: df_new[c] = None
    df_new = df_new[CANON]

    if df_current.empty:
        take = df_new.head(needed)
        return take.copy(), needed - len(take)

    # dedupe by URL across what we already have
    have = set(df_current["job_url"].dropna())
    keep_rows = []
    for _, r in df_new.iterrows():
        u = r.get("job_url")
        if u and u not in have:
            keep_rows.append(r)
            have.add(u)
        if len(keep_rows) >= needed:
            break
    if keep_rows:
        df_current = pd.concat([df_current, pd.DataFrame(keep_rows, columns=CANON)], ignore_index=True)
        needed = max(0, needed - len(keep_rows))
    return df_current, needed

# RUN: accumulate 25 per source (LinkedIn / Indeed / Generic)
li_needed = TARGET_PER_SOURCE
in_needed = TARGET_PER_SOURCE
gg_needed = TARGET_PER_SOURCE

df_li_all = pd.DataFrame(columns=CANON)
df_in_all = pd.DataFrame(columns=CANON)
df_gg_all = pd.DataFrame(columns=CANON)

print("Collecting up to", TARGET_PER_SOURCE, "per source (LinkedIn/Indeed/Generic)…")

# Iterate over your QUERIES to fill each bucket up to target
for role, loc, n in QUERIES:
    if li_needed > 0:
        df_li_try = fetch_domain(role, loc, min(n, li_needed), domain="linkedin.com")
        if not df_li_try.empty:
            df_li_try["source"] = "LinkedIn (via GoogleJobs/SerpAPI)"
            df_li_all, li_needed = _accumulate_until(df_li_all, df_li_try, li_needed)

    if in_needed > 0:
        df_in_try = fetch_domain(role, loc, min(n, in_needed), domain="indeed.com")
        if not df_in_try.empty:
            df_in_try["source"] = "Indeed (via GoogleJobs/SerpAPI)"
            df_in_all, in_needed = _accumulate_until(df_in_all, df_in_try, in_needed)

    if gg_needed > 0:
        df_gg_try = fetch_generic(role, loc, min(n, gg_needed))
        if not df_gg_try.empty:
            df_gg_all, gg_needed = _accumulate_until(df_gg_all, df_gg_try, gg_needed)

    # Stop early if we already filled all buckets
    if li_needed == 0 and in_needed == 0 and gg_needed == 0:
        break

# If any bucket still short, you can optionally loop again over QUERIES with broader settings
print(f"Filled — LinkedIn: {len(df_li_all)} / {TARGET_PER_SOURCE}, Indeed: {len(df_in_all)} / {TARGET_PER_SOURCE}, Generic: {len(df_gg_all)} / {TARGET_PER_SOURCE}")

# Combine WITHOUT cross-dedup (preserve up to 25 per source even if overlaps exist)
df_75 = pd.concat([df_li_all, df_in_all, df_gg_all], ignore_index=True, sort=False)

# Make sure canonical columns are present
for c in CANON:
    if c not in df_75.columns: df_75[c] = None
df_75 = df_75[CANON]

print("Total rows in combined set:", len(df_75))
display(df_75.head(20))

# ---- Save CSV and trigger download in Colab ----
if not CSV_PATH:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_role = re.sub(r"[^A-Za-z0-9]+","_", ROLE_FOR_CSV_NAME).strip("_") or "jobs"
    CSV_PATH = f"jobs_{safe_role}_{stamp}_L25_I25_G25.csv"

df_75.to_csv(CSV_PATH, index=False)
print("Saved CSV:", CSV_PATH)

# Auto-download if running in Google Colab
try:
    from google.colab import files
    files.download(CSV_PATH)
except Exception:
    print("Download: If not in Colab, retrieve the file from your working directory.")


[linkedin.com] Attempts: 200 site:linkedin.com Data Analyst [New York, NY] | 200 site:linkedin.com Data Analyst [United States] | 200 site:linkedin.com Data Analyst [no-loc] | 200 Data Analyst [New York, NY] | 200 Data Analyst [United States] | 200 Data Analyst [no-loc]
[linkedin.com] Found: 2
[indeed.com] Attempts: 200 site:indeed.com Data Analyst [New York, NY] | 200 site:indeed.com Data Analyst [United States] | 200 site:indeed.com Data Analyst [no-loc] | 200 Data Analyst [New York, NY] | 200 Data Analyst [United States] | 200 Data Analyst [no-loc]
[indeed.com] Found: 7
[linkedin.com] Attempts: 200 site:linkedin.com Data Engineer remote [United States] | 200 site:linkedin.com Data Engineer remote [no-loc] | 200 Data Engineer remote [United States] | 200 Data Engineer remote [United States] | 200 Data Engineer remote [no-loc]
[linkedin.com] Found: 0
[indeed.com] Attempts: 200 site:indeed.com Data Engineer remote [United States] | 200 site:indeed.com Data Engineer remote [no-loc] | 20

,post_date,company,job_title,position,location,job_url,source,description
0,3 days ago,Capital Rx,Network Pricing Data Analyst,Network Pricing Data Analyst,Anywhere,https://www.linkedin.com/jobs/view/network-pri...,LinkedIn (via GoogleJobs/SerpAPI),About Capital Rx Capital Rx is a health techno...
1,10 hours ago,Harvey Nash,"Data Analyst - New Haven, CT (Hybrid)","Data Analyst - New Haven, CT (Hybrid)","New Haven, CT",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn (via GoogleJobs/SerpAPI),"Job Title: Data Analyst Location: New Haven, C..."
2,6 days ago,Montefiore Einstein,Data Analyst,Data Analyst,"Bronx, NY",https://www.indeed.com/viewjob?jk=c4cf7b0726f8...,Indeed (via GoogleJobs/SerpAPI),"City/State: Bronx, New York Grant Funded: Yes ..."
3,6 days ago,American Express,Senior Analyst - Data Analytics,Senior Analyst - Data Analytics,"New York, NY",https://www.indeed.com/viewjob?jk=b71a9944b76c...,Indeed (via GoogleJobs/SerpAPI),"At American Express, our culture is built on a..."
4,None,PayPal,"Analyst, Data Analytics","Analyst, Data Analytics","New York, NY",https://www.indeed.com/viewjob?jk=65d367fd99ff...,Indeed (via GoogleJobs/SerpAPI),The Company PayPal has been revolutionizing co...
5,5 days ago,Guardian Life Insurance Company,Customer Data Management Analyst,Customer Data Management Analyst,"New York, NY",https://www.indeed.com/viewjob?jk=772eb7ff6af0...,Indeed (via GoogleJobs/SerpAPI),"Every day, Guardian helps our 29 million custo..."
6,None,Meta,Data Analyst - People Analytics,Data Analyst - People Analytics,"New York, NY",https://www.indeed.com/viewjob?jk=7a36f733e4fc...,Indeed (via GoogleJobs/SerpAPI),Data Analysts in People Analytics drive the bu...
7,13 days ago,ZYWAVE INC.,Data Analyst - Remote,Data Analyst - Remote,Anywhere,https://www.indeed.com/viewjob?jk=01ba14405e62...,Indeed (via GoogleJobs/SerpAPI),Position Purpose: • Assist Zywave partners wit...
8,6 days ago,Cellhub,Data Analyst - Junior,Data Analyst - Junior,"Hicksville, NY",https://www.indeed.com/viewjob?jk=2b85e3399082...,Indeed (via GoogleJobs/SerpAPI),Job Overview We are seeking a detail-oriented ...
9,None,Scalepex,AWS Data Engineer - Fully Remote - US Only,AWS Data Engineer - Fully Remote - US Only,Anywhere,https://www.indeed.com/viewjob?jk=ef73e4bc9fad...,Indeed (via GoogleJobs/SerpAPI),❋ Why Scalepex? Scalepex is a dynamic services...


Saved CSV: jobs_Data_Analyst_20250922_232633_L25_I25_G25.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>